# March Machine Learning Mania 2026

|No| Name  | Notebook |
| --- | ------ | ------- |
| 00 | Dataset |  https://www.kaggle.com/competitions/march-machine-learning-mania-2026/data |
| 01 | Understand Data Structure  | https://www.kaggle.com/code/rudraprasadbhuyan/ncaa-p1-dataset-overview-structure-understanding/ |
| 02 | Create a baseline model only on Male rqd. 4 datasets |  |
| | |  |

In [257]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

After this notebook, We understand:

- Which data files are most imp.
- How to create features
- What is our main goal 
- First submission

# Imports only 4 file

In [258]:
name_path_dict = {
    
    # Male data
    'MTeam_df': 'MTeams.csv',
    'MSeasons_df': 'MSeasons.csv',
    'MTourneySeed_df': 'MNCAATourneySeeds.csv',
    'MRegularSeasonCompactResults_df': 'MRegularSeasonCompactResults.csv',
    'MTourneyResults_df': 'MNCAATourneyCompactResults.csv',

    
    # Female data
    'WTeams_df': 'WTeams.csv',
    'WSeasons_df': 'WSeasons.csv',
    'WTourneySeed_df': 'WNCAATourneySeeds.csv',
    'WRegularSeasonCompactResults_df': 'WRegularSeasonCompactResults.csv',
    'WTourneyResults_df': 'WNCAATourneyCompactResults.csv',

    # Misc
    'SampleSubmissionStage1_df': 'SampleSubmissionStage1.csv',
    'SampleSubmissionStage2_df': 'SampleSubmissionStage2.csv',

}


# Root file
data_file_path = r"C:\Users\Rudra\Desktop\kaggle\NCAA\data"

# Load only required files
regular = pd.read_csv(os.path.join(data_file_path, "MRegularSeasonCompactResults.csv"))
tourney = pd.read_csv(os.path.join(data_file_path,"MNCAATourneyCompactResults.csv"))
seeds = pd.read_csv(os.path.join(data_file_path,"MNCAATourneySeeds.csv"))
submission = pd.read_csv(os.path.join(data_file_path,"SampleSubmissionStage1.csv"))

# Sample View
display(regular.sample(3))
display(tourney.sample(3))
display(seeds.sample(3))
display(submission.sample(3))

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
88830,2006,37,1458,82,1453,62,H,0
183337,2024,40,1276,83,1185,66,H,0
58634,1999,86,1104,67,1116,60,A,0


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
2467,2024,136,1332,87,1376,73,N,0
42,1985,139,1210,70,1393,53,N,0
1089,2002,137,1143,82,1335,75,N,0


,Season,Seed,TeamID
1411,2006,Z14,1293
2329,2021,Y08,1260
2123,2017,Y08,1274


,ID,Pred
118913,2022_3339_3417,0.5
381646,2024_3362_3378,0.5
463810,2025_3133_3421,0.5


# Create Win Counts

In [259]:
# wins
wins = regular.groupby(['Season', 'WTeamID']).size().reset_index(name='wins')

# losses
losses = regular.groupby(['Season', 'LTeamID']).size().reset_index(name='losses')

# rename for merge
wins = wins.rename(columns={"WTeamID": 'TeamID'})
losses = losses.rename(columns={'LTeamID': 'TeamID'})

# Merge
team_stats = pd.merge(
    wins, losses,
    how="outer", on=["Season", 'TeamID']
).fillna(0)

# total games
team_stats["TotalGames"] = team_stats["wins"] + team_stats["losses"]

# Win %page
team_stats['WinPct'] = team_stats['wins'] / team_stats['TotalGames']

display(team_stats.sample(3))

,Season,TeamID,wins,losses,TotalGames,WinPct
12266,2022,1438,19.0,13.0,32.0,0.593750
3171,1995,1405,14.0,11.0,25.0,0.560000
11846,2021,1373,12.0,5.0,17.0,0.705882


# Clean Seeds

In [260]:
seeds['SeedNum'] = seeds['Seed'].str[1:3].astype(int)

display(seeds.sample(3))

,Season,Seed,TeamID,SeedNum
1831,2013,X05,1433,5
301,1989,Y14,1166,14
1281,2004,Z15,1434,15


#  Prepare Tournament Training Data

In [261]:
# smaller team id -> team 1
tourney['Team1'] = tourney[['WTeamID', 'LTeamID']].min(axis=1)
# larger team id -> team 2
tourney['Team2'] = tourney[['WTeamID', 'LTeamID']].max(axis=1)

# if team1 won then 1 else 0
tourney['Team1Win'] = (tourney['WTeamID'] == tourney['Team1']).astype(int)

train = tourney[['Season', 'Team1', 'Team2', 'Team1Win']]
display(train.sample(3))
print(train.shape)

,Season,Team1,Team2,Team1Win
1290,2005,1150,1301,0
1324,2005,1314,1458,1
699,1996,1246,1363,1


(2585, 4)


#  Features

In [262]:
def add_features(df):

    df = df.merge(
        team_stats[['Season','TeamID','WinPct']],
        left_on=['Season','Team1'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'WinPct':'Team1_WinPct'}).drop('TeamID', axis=1)

    df = df.merge(
        team_stats[['Season','TeamID','WinPct']],
        left_on=['Season','Team2'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'WinPct':'Team2_WinPct'}).drop('TeamID', axis=1)

    df = df.merge(
        seeds[['Season','TeamID','SeedNum']],
        left_on=['Season','Team1'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'SeedNum':'Team1_Seed'}).drop('TeamID', axis=1)

    df = df.merge(
        seeds[['Season','TeamID','SeedNum']],
        left_on=['Season','Team2'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'SeedNum':'Team2_Seed'}).drop('TeamID', axis=1)

    return df


In [263]:
train = add_features(train)

# Create Final Model Features

In [265]:
train['WinPct_Diff'] = train['Team1_WinPct'] - train['Team2_WinPct']

# lower seed = strong team so (Team2 - Team1)
train['Seed_Diff'] = train['Team2_Seed'] - train['Team1_Seed'] 

train

,Season,Team1,Team2,Team1Win,Team1_WinPct,Team2_WinPct,Team1_Seed,Team2_Seed,WinPct_Diff,Seed_Diff
0,1985,1116,1234,1,0.636364,0.666667,9,8,-0.030303,-1
1,1985,1120,1345,1,0.620690,0.680000,11,6,-0.059310,-5
2,1985,1207,1250,1,0.925926,0.379310,1,16,0.546616,15
3,1985,1229,1425,1,0.740741,0.678571,9,8,0.062169,-1
4,1985,1242,1325,1,0.766667,0.740741,3,14,0.025926,11
...,...,...,...,...,...,...,...,...,...,...
2580,2025,1120,1277,1,0.848485,0.818182,1,2,0.030303,1
2581,2025,1222,1397,1,0.882353,0.794118,1,2,0.088235,1
2582,2025,1120,1196,0,0.848485,0.882353,1,1,-0.033868,0
2583,2025,1181,1222,0,0.911765,0.882353,1,1,0.029412,0


# Baseline Model

In [266]:
train.columns

Index(['Season', 'Team1', 'Team2', 'Team1Win', 'Team1_WinPct', 'Team2_WinPct',
       'Team1_Seed', 'Team2_Seed', 'WinPct_Diff', 'Seed_Diff'],
      dtype='object')

- Instead of give Team1Pct, Team2Pct value
- We provide the difference
- Because our goal is `Probability that Team1 beats Team2`

In [267]:
X = train[['WinPct_Diff', 'Seed_Diff']]
y = train['Team1Win']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression()
model.fit(X_train, y_train)

preds = model.predict_proba(X_val)[:, 1]

print(f'LogLoss {log_loss(y_val, preds)}')

LogLoss 0.5322181507855879


# Submission

In [268]:
submission.head(3)

,ID,Pred
0,2022_1101_1102,0.5
1,2022_1101_1103,0.5
2,2022_1101_1104,0.5


In [269]:
def create_id(df):
    df[['Season','Team1','Team2']] = df['ID'].str.split('_', expand=True)

    df['Season'] = df['Season'].astype(int)
    df['Team1'] = df['Team1'].astype(int)
    df['Team2'] = df['Team2'].astype(int)

    return df

In [ ]:
submission = create_id(submission)
submission = add_features(submission)

# only men for now
submission_men = submission[
    (submission['Team1'] < 2000) &
    (submission['Team2'] < 2000)
].copy()


submission['WinPct_Diff'] = submission['Team1_WinPct'] - submission['Team2_WinPct']
submission['Seed_Diff'] = submission['Team2_Seed'] - submission['Team1_Seed']

submission[['WinPct_Diff','Seed_Diff']] = \
submission[['WinPct_Diff','Seed_Diff']].fillna(0)

display(submission.sample(3))

In [ ]:
submission['Pred'] = model.predict_proba(
    submission[['WinPct_Diff','Seed_Diff']]
)[:,1]

submission[['ID','Pred']]

In [277]:
submission[['ID','Pred']].to_csv("baseline_submission.csv", index=False)